# `noaa_coops`: New Features

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GClunies/noaa_coops/blob/main/examples/noaa_coops_new_features_tutorial.ipynb)

`noaa_coops` is a Python wrapper around NOAA's CO-OPS APIs for tide, current, and water level data.

This notebook walks through what's new: improved pagination, `daily_max_min` support, derived products (DPAPI), and enhanced metadata access.

## Setup

This branch hasn't merged upstream yet, so `pip install noaa_coops` won't include these features. Run the cell below once — it installs this branch. `%pip` installs into whatever kernel is currently running this notebook, so pick any Python 3.10+ kernel and just run the cells top to bottom — no separate environment setup required.

Once this lands upstream, swap the install line for `%pip install -q noaa_coops`.

In [ ]:
%pip install -q "noaa_coops @ git+https://github.com/staree14/noaa_coops_dev.git@gsoc-2026-final"
print("Setup complete — noaa_coops installed.")

In [ ]:
from noaa_coops import Station

## What's new, at a glance

- Improved pagination across NOAA's per-product date-range caps (no more incorrect chunking for some products' multi-year requests)
- `daily_max_min` product support, with `max_min_type` to filter to just `"max"` or `"min"`
- Derived products (DPAPI) — HTF flooding counts, extreme water levels, sea level trends and projections
- Cleaner, more reliable metadata (tide offsets, current bins, datums, `center_bin_1_dist` for current stations)

### Pagination that works correctly!

NOAA caps how much date range a single request can cover, and the cap varies by product (31 days for `water_level`, 365 for `hourly_height`, up to 10 years for `daily_mean`). Requesting a wider range used to mean chunking it into hardcoded blocks which weren't accurate for every product. Now `get_data()` does the right chunking, and prodcuts data is returned at a higher speed 


In [ ]:
# A 3-year daily_mean range — past the 31-day cap most products carry, one call.
p = Station("9063038")  # Erie, Lake Erie
df= p.get_data(
    begin_date="20200630",
    end_date="20230630",
    datum="IGLD",
    time_zone="gmt",
    interval="hilo",
    product="daily_mean",
    units="english",
)

print(df.attrs.get("missing_blocks", []))  # empty = clean stitch across blocks
df

### `daily_max_min` support

Returns NOAA's daily extrema instead of the standard `v`/`s`/`f`/`q` shape — each day contributes a `max` row and a `min` row, distinguished by `record_type`. `interval` defaults to `"h"` if you don't set it, so 6-minute and hourly data never get mixed in the same DataFrame.

In [ ]:
# Example 1: interval="6" pulls both max and min at 6-minute-derived resolution
p = Station("9491094")
df = p.get_data(begin_date="20170101", end_date="20170102",
                 datum="STND", product="daily_max_min",
                 interval=6, units="english", time_zone="gmt")
df

In [ ]:
# Example 2: max_min_type="max" — only the daily highs, useful when you only care about peak levels (e.g. flood risk)
q = Station("9447130")
df2 = q.get_data(begin_date="20150101", end_date="20150110",
                  datum="MLLW", product="daily_max_min",
                  max_min_type="max", units="metric") # defaults to hourly
df2 

### Derived products via DPAPI

Some NOAA products (high tidal flooding counts/outlooks, extreme water levels, sea level trends and rise projections) live on a separate API (DPAPI) — now accessible through the same `Station` interface via `get_derived_product()`. Below are examples across the three families: HTF flooding, sea level trends/projections, and extreme water levels.

In [ ]:
a = Station("9447130")
a.get_derived_product(product="htf_daily", start_date="20180101", end_date="20180630")

`htf_monthly` returns the daily flooding flags up into monthly counts by severity (`minCount`/`modCount`/`majCount`) — useful for flooding frequency trends.

In [ ]:
# Eagle Point, TX — Gulf coast station with real minor-flooding counts in this window
htf = Station("8771013")
htf.get_derived_product(product="htf_monthly", start_date="20190601", end_date="20191231")


#### Sea level trends & projections

In [ ]:
a = Station("9461380")
a.get_derived_product(product="sea_level_trends")

In [ ]:
a = Station("9461380")
a.get_derived_product(product="sea_level_trends",detail="seasonal_cycle")

`slr_projections` gives forward-looking rise estimates instead of the trend above — filter by `scenario` (`"low"` through `"extreme"`), `projection_year`, and `report_year`.

In [ ]:
# Adak, AK — "high" scenario projection for 2050, from NOAA's 2022 report
adak = Station("9461380")
slr = adak.get_derived_product(
    product="slr_projections", projection_year=2050, report_year=2022, scenario="high"
)
slr[["stationName", "scenario", "projectionYear", "reportYear", "projectionRsl", "projectionCiLow", "projectionCiHigh"]]

#### RFA extreme water levels

Returns exploded, repeated station-attribute rows so that every return-period row still carries its own station identity — convenient for concatenating results across multiple stations without a separate join.

In [ ]:
b= Station("1611347")
b.get_derived_product(product="rfa_extreme_water_levels")

`extreme_water_levels` is the related single-station endpoint behind the RFA return-period
statistics above. Filter with `level_type="high"` or `"low"`.

In [ ]:
extremes = a.get_derived_product(product="extreme_water_levels", units="metric", level_type="low")
extremes[["type", "date", "status"]] 
# uncomment this next line to see full DataFrame
# extremes 

### Cleaner metadata access

Fields like tide prediction offsets and current bins used to be inconsistent across station types. Access is now consistent regardless of station quirks.

In [ ]:
# Station with multiple current bins
u = Station("ACT0921")
u.current_pred_offsets_by_bin

In [ ]:
# Subordinate station, null datums but has tidePredOffsets
s = Station("8557863")
s.tide_pred_offsets

In [ ]:
# Subordinate station, with datums and tidePredOffsets
t = Station("8720135")  
t.tide_pred_offsets

In [ ]:
# Harmonic station, has datums and no tidePredOffsets
q = Station("8570280")
q.datums

### Where to go next

- Full docs: in README.md
- Deferred to a later phase: `peakwaterlevels`, `extremewaterlevels` sub-endpoint, `htf_outlook`, bounding-box queries 